# 12 · Support Vector Machine (SVM) Classification
Train an RBF-kernel SVM classifier on UrbanSound8K training folds (1–9) with 9-fold grouped cross-validation (LeaveOneGroupOut), tune hyperparameters (C, gamma), evaluate on held-out test Fold 10, and export candidate artifacts.

In [1]:
# Cell 1 – interpreter setup & imports
import sys
from pathlib import Path
import time
import hashlib

print('sys.executable:', sys.executable)
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('REPO_ROOT     :', REPO_ROOT)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import LeaveOneGroupOut, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

print('Successfully imported sklearn, joblib, numpy, pandas, and matplotlib')

sys.executable: C:\Users\LENOVO\miniconda3\envs\citypulse\python.exe
REPO_ROOT     : C:\Users\LENOVO\Desktop\CityPulse


Successfully imported sklearn, joblib, numpy, pandas, and matplotlib


In [2]:
# Cell 2 – verify sha256 of inputs, load data and build train/test sets
def compute_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

labels_path = REPO_ROOT / 'data/processed/labels.csv'
features_path = REPO_ROOT / 'data/processed/audio_features.npy'
split_path = REPO_ROOT / 'data/processed/split_indices.npz'

labels_sha = compute_sha256(labels_path)
features_sha = compute_sha256(features_path)
split_sha = compute_sha256(split_path)

print(f'labels.csv sha256        : {labels_sha}')
print(f'audio_features.npy sha256: {features_sha}')
print(f'split_indices.npz sha256 : {split_sha}')

EXPECTED_LABELS_SHA = '8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584dc8e86b9180d5f854'
EXPECTED_FEAT_SHA   = 'ed16a16f2a9e5c90aa61e925b461e0bce053c4654f1cc22a0812264630b1c73a'
EXPECTED_SPLIT_SHA  = '2c0cdd397d4d9a7c3decf630a214cae2f55e7bd8c0ead5693a79d36b4cc8b10b'

assert labels_sha == EXPECTED_LABELS_SHA, f'labels.csv SHA mismatch: {labels_sha}'
assert features_sha == EXPECTED_FEAT_SHA, f'audio_features.npy SHA mismatch: {features_sha}'
assert split_sha == EXPECTED_SPLIT_SHA, f'split_indices.npz SHA mismatch: {split_sha}'

labels = pd.read_csv(labels_path)
X = np.load(features_path)

with np.load(split_path) as split_data:
    train_idx = split_data['train_idx']
    test_idx = split_data['test_idx']
    esc_idx = split_data['esc_idx']

X_train = X[train_idx]
y_train = labels.loc[train_idx, 'app_class'].to_numpy()
groups_train = labels.loc[train_idx, 'fold'].to_numpy()

X_test = X[test_idx]
y_test = labels.loc[test_idx, 'app_class'].to_numpy()

print(f'X_train shape     : {X_train.shape}')
print(f'y_train shape     : {y_train.shape}')
print(f'groups_train shape: {groups_train.shape}')
print(f'X_test shape      : {X_test.shape}')
print(f'y_test shape      : {y_test.shape}')

assert X_train.shape == (5753, 128), f'Expected X_train shape (5753, 128), got {X_train.shape}'
assert X_test.shape == (605, 128), f'Expected X_test shape (605, 128), got {X_test.shape}'
assert len(y_train) == 5753, f'Expected 5753 in y_train, got {len(y_train)}'
assert len(groups_train) == 5753, f'Expected 5753 in groups_train, got {len(groups_train)}'
assert len(y_test) == 605, f'Expected 605 in y_test, got {len(y_test)}'

print('Cell 2 SHA256 asserts and loads passed ✓')

labels.csv sha256        : 8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584dc8e86b9180d5f854
audio_features.npy sha256: ed16a16f2a9e5c90aa61e925b461e0bce053c4654f1cc22a0812264630b1c73a
split_indices.npz sha256 : 2c0cdd397d4d9a7c3decf630a214cae2f55e7bd8c0ead5693a79d36b4cc8b10b
X_train shape     : (5753, 128)
y_train shape     : (5753,)
groups_train shape: (5753,)
X_test shape      : (605, 128)
y_test shape      : (605,)
Cell 2 SHA256 asserts and loads passed ✓


In [3]:
# Cell 3 – Pipeline and GridSearchCV with LeaveOneGroupOut (9 folds)
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='rbf', probability=True, random_state=42))
])

param_grid = {
    'svc__C': [1, 10, 100],
    'svc__gamma': ['scale', 0.01, 0.001]
}

cv = LeaveOneGroupOut()
grid_search = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=1,
    refit=True,
    verbose=0
)

print('Starting GridSearchCV on train folds 1–9...')
t0 = time.perf_counter()
grid_search.fit(X_train, y_train, groups=groups_train)
elapsed = time.perf_counter() - t0

print(f'GridSearchCV fit completed in {elapsed:.2f} s ({elapsed/60:.2f} min)')

cv_results_df = pd.DataFrame(grid_search.cv_results_)
table_cols = ['params', 'mean_test_score', 'std_test_score', 'rank_test_score']
cv_table = cv_results_df[table_cols].sort_values('rank_test_score').rename(columns={'rank_test_score': 'rank'})

print('\n=== Full CV Results Table (sorted by rank) ===')
print(cv_table.to_string(index=False))

print(f'\nBest parameters: {grid_search.best_params_}')
print(f'Best CV accuracy: {grid_search.best_score_:.4f} ({grid_search.best_score_*100:.2f}%)')

assert grid_search.n_splits_ == 9, f'Expected 9 CV splits, got {grid_search.n_splits_}'
print('Cell 3 asserts passed: 9-fold grouped CV verified ✓')

Starting GridSearchCV on train folds 1–9...


C:\Users\LENOVO\miniconda3\envs\citypulse\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\LENOVO\miniconda3\envs\citypulse\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\LENOVO\miniconda3\envs\citypulse\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\LENOVO\miniconda3\envs\citypulse\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\LENOVO\miniconda3\envs\citypulse\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


... (74 outputs left out)

C:\Users\LENOVO\miniconda3\envs\citypulse\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\LENOVO\miniconda3\envs\citypulse\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\LENOVO\miniconda3\envs\citypulse\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


GridSearchCV fit completed in 492.09 s (8.20 min)

=== Full CV Results Table (sorted by rank) ===
                                params  mean_test_score  std_test_score  rank
 {'svc__C': 10, 'svc__gamma': 'scale'}         0.677381        0.065950     1
  {'svc__C': 100, 'svc__gamma': 0.001}         0.676773        0.068425     2
    {'svc__C': 10, 'svc__gamma': 0.01}         0.676267        0.064904     3
     {'svc__C': 1, 'svc__gamma': 0.01}         0.673674        0.054711     4
  {'svc__C': 1, 'svc__gamma': 'scale'}         0.667905        0.063148     5
   {'svc__C': 10, 'svc__gamma': 0.001}         0.658570        0.071477     6
{'svc__C': 100, 'svc__gamma': 'scale'}         0.639384        0.049642     7
   {'svc__C': 100, 'svc__gamma': 0.01}         0.637300        0.045680     8
    {'svc__C': 1, 'svc__gamma': 0.001}         0.616384        0.064543     9

Best parameters: {'svc__C': 10, 'svc__gamma': 'scale'}
Best CV accuracy: 0.6774 (67.74%)
Cell 3 asserts passed: 9-fold gr

In [4]:
# Cell 4 – single final evaluation on test Fold 10
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

test_accuracy = float(accuracy_score(y_test, y_pred))
majority_baseline = 0.3240
test_macro_f1 = float(f1_score(y_test, y_pred, average='macro'))

print(f'Test Accuracy        : {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')
print(f'Majority Baseline Acc: {majority_baseline:.4f} ({majority_baseline*100:.2f}%)')
print(f'Test Macro-F1        : {test_macro_f1:.4f}')

print('\n=== Classification Report (Test Fold 10) ===')
report_text = classification_report(y_test, y_pred, digits=4)
print(report_text)

assert test_accuracy > majority_baseline, f'Test accuracy {test_accuracy:.4f} <= baseline {majority_baseline:.4f}'
print('Cell 4 asserts passed: test accuracy beats majority baseline ✓')

Test Accuracy        : 0.7967 (79.67%)
Majority Baseline Acc: 0.3240 (32.40%)
Test Macro-F1        : 0.7780

=== Classification Report (Test Fold 10) ===
                   precision    recall  f1-score   support

     construction     0.8333    0.8673    0.8500       196
             hvac     0.6303    0.7500    0.6849       100
music_loudspeaker     0.8131    0.8700    0.8406       100
            siren     0.8103    0.5663    0.6667        83
     traffic_horn     0.8803    0.8175    0.8477       126

         accuracy                         0.7967       605
        macro avg     0.7935    0.7742    0.7780       605
     weighted avg     0.8031    0.7967    0.7955       605

Cell 4 asserts passed: test accuracy beats majority baseline ✓


In [5]:
# Cell 5 – confusion matrix, visualization, lowest F1, and largest confusion pair
classes = sorted(np.unique(y_test))
cm = confusion_matrix(y_test, y_pred, labels=classes)

cm_df = pd.DataFrame(cm, index=classes, columns=classes)
cm_df.index.name = 'True'
cm_df.columns.name = 'Predicted'

print('=== Confusion Matrix (Fold 10) ===')
print(cm_df.to_string())

fig_dir = REPO_ROOT / 'reports/figures'
fig_dir.mkdir(parents=True, exist_ok=True)
fig_path = fig_dir / 'svm_confusion_matrix.png'

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
disp.plot(ax=ax, cmap='Blues', colorbar=True)
plt.title('SVM RBF - Confusion Matrix (Test Fold 10)')
plt.tight_layout()
plt.savefig(fig_path, dpi=150)
plt.close(fig)
print(f'\nSaved confusion matrix plot to: {fig_path}')

# Lowest F1 class
report_dict = classification_report(y_test, y_pred, output_dict=True)
per_class_f1 = {cls: report_dict[cls]['f1-score'] for cls in classes}
lowest_f1_class = min(per_class_f1, key=per_class_f1.get)
print(f'Class with lowest F1: {lowest_f1_class} (F1 = {per_class_f1[lowest_f1_class]:.4f})')

# Largest off-diagonal confusion pair
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)
max_off_val = int(np.max(cm_off))
i_max, j_max = np.unravel_index(np.argmax(cm_off), cm_off.shape)
true_confused = classes[i_max]
pred_confused = classes[j_max]
print(f'Largest off-diagonal confusion pair: True={true_confused} -> Pred={pred_confused} ({max_off_val} clips)')
print('Cell 5 completed ✓')

=== Confusion Matrix (Fold 10) ===
Predicted          construction  hvac  music_loudspeaker  siren  traffic_horn
True                                                                         
construction                170    13                  4      6             3
hvac                         13    75                  3      1             8
music_loudspeaker             6     2                 87      4             1
siren                         4    20                 10     47             2
traffic_horn                 11     9                  3      0           103



Saved confusion matrix plot to: C:\Users\LENOVO\Desktop\CityPulse\reports\figures\svm_confusion_matrix.png
Class with lowest F1: siren (F1 = 0.6667)
Largest off-diagonal confusion pair: True=siren -> Pred=hvac (20 clips)
Cell 5 completed ✓


In [6]:
# Cell 6 – save candidate model and cv_results, reload and verify
models_dir = REPO_ROOT / 'models'
models_dir.mkdir(parents=True, exist_ok=True)
model_path = models_dir / 'svm_candidate.joblib'
joblib.dump(best_model, model_path)
print(f'Saved candidate model to: {model_path}')

reports_dir = REPO_ROOT / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)
cv_csv_path = reports_dir / 'svm_cv_results.csv'
cv_results_df.to_csv(cv_csv_path, index=False)
print(f'Saved cv_results to: {cv_csv_path}')

# Reload model and verify predictions
reloaded_model = joblib.load(model_path)
reloaded_pred = reloaded_model.predict(X_test)

assert np.array_equal(reloaded_pred, y_pred), 'Reloaded model predictions do not equal in-memory predictions'
print('Reloaded model predictions match in-memory predictions exactly ✓')

model_sha = compute_sha256(model_path)
cv_sha = compute_sha256(cv_csv_path)

print(f'\nsvm_candidate.joblib sha256: {model_sha}')
print(f'svm_cv_results.csv   sha256: {cv_sha}')
print('Cell 6 asserts passed ✓')

Saved candidate model to: C:\Users\LENOVO\Desktop\CityPulse\models\svm_candidate.joblib
Saved cv_results to: C:\Users\LENOVO\Desktop\CityPulse\reports\svm_cv_results.csv


Reloaded model predictions match in-memory predictions exactly ✓

svm_candidate.joblib sha256: 3e798d576f2d30e7bb4ae19345879598527d07f33385d9b9f1529d433de3d9e2
svm_cv_results.csv   sha256: 8788f726e3f65dc095fddfb167f1229125f5ddd2483dce756787109bf02a08cf
Cell 6 asserts passed ✓
